# Analisi statistica e critica delle campagne approvate

Analisi completamente offline delle nove combinazioni: nessuna connessione DB, ingestione o riesecuzione delle query. Gli originali sono aperti solo in lettura. Il catalogo proviene da `config/demo.yaml`; le impostazioni DEMO live non vengono utilizzate.

100K deriva dal controllo funzionale preliminare approvato; 1M e FULL da campagne successive. Le dimensioni sono prefissi ordinati del CSV, non campioni casuali. Le 20 misure sono ripetizioni sequenziali nella stessa sessione: non stimano la variabilità fra esecuzioni complete indipendenti. Nessun p-value, intervallo di confidenza o dichiarazione di significatività.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src').is_dir() and (p/'config').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import display, HTML, SVG
from src.statistical_analysis import load_analysis_catalog, analyze_campaigns, export_analysis
from src.statistical_plots import distribution_chart, growth_chart
from src.presentation import html_table
campaigns, results_root = load_analysis_catalog(ROOT/'config/demo.yaml')
analysis = analyze_campaigns(campaigns, results_root)
def table(name, columns):
    display(HTML(html_table(analysis[name], columns)))
print('Serie query:', len(analysis['queries']), '| misure:', sum(g['n'] for g in analysis['queries']))
print('Sorgenti verificate:', len(analysis['source_hashes']))


## Vista sintetica iniziale
Cinque grafici principali per una dimensione selezionata: ingestione, lookup Q1, Q2/Q3 prima e dopo gli indici e aggregazione Q4. Cambiare `DIMENSIONE_SINTESI` per 100K, 1M o FULL.
I restanti **22 grafici** rimangono nelle sezioni successive, attivabili con `MOSTRA_APPROFONDIMENTI=True`: totale **27**. Tutti derivano soltanto dagli artefatti storici FULL; nessuna misura LIVE.


In [ ]:
from src.history import read_catalog
from src.presentation import historical_charts
DIMENSIONE_SINTESI = 5819079
MOSTRA_APPROFONDIMENTI = False
summaries = read_catalog(campaigns, results_root)
for svg in historical_charts(summaries, DIMENSIONE_SINTESI).values():
    display(SVG(svg))


## Controlli e convenzioni

Si verificano manifest completati, protocollo comune, riferimenti e snapshot coerenti. Dai raw si controllano identità, ordine delle 23 iterazioni, 3 warm-up, 20 misure, parametri e digest. Q4 mantiene la tolleranza numerica approvata tramite gli snapshot: un digest differente per arrotondamento non dimostra una differenza logica. Le cinque statistiche storiche sono ricalcolate e confrontate con i CSV originali.

Tempi in **millisecondi**, deviazione standard **campionaria (n-1)**. Quantili con interpolazione lineare nella posizione (n-1)p; MAD = mediana degli scarti assoluti dalla mediana, senza fattore di scala. CV = 100s/media. I punti oltre 1,5 IQR sono soltanto segnalati e restano inclusi.


In [ ]:
table('queries', ['database','dataset_size','query_id','index_state','category','n','mean_ms','median_ms','sample_stddev_ms','min_ms','max_ms','iqr_ms','mad_ms','cv_pct','tukey_flags'])

## Distribuzione e ordine delle misure

Selezionare dimensione, query e scenario. Baffi min/max, box IQR, mediana e **tutti i 20 punti**. La traccia usa l'ordine locale delle sessioni, non un tempo comune fra DB. Lag-1 e confronto delle due metà sono indizi descrittivi di variazione temporale, non test di indipendenza. Correlazione nulla non prova indipendenza.


## Approfondimenti: altre dimensioni
Ingestione, Q1-Q4 e confronto baseline/indici: ogni grafico mantiene separati i dataset. Le barre mostrano medie; la dispersione delle query rimane nelle tabelle e nei grafici dei punti. Ingestione: un solo evento per motore e dimensione.

In [ ]:
if MOSTRA_APPROFONDIMENTI is True:
    for size in (100000, 1000000, 5819079):
        if size != DIMENSIONE_SINTESI:
            for svg in historical_charts(summaries, size).values():
                display(SVG(svg))

else:
    print('Operazione disabilitata: MOSTRA_APPROFONDIMENTI=False.')


In [ ]:
if MOSTRA_APPROFONDIMENTI is True:
    DIMENSIONE = 5819079
    QUERY = 'Q3'
    SCENARIO = 'secondary'
    selected = [g for g in analysis['queries'] if (g['dataset_size'],g['query_id'],g['index_state']) == (DIMENSIONE,QUERY,SCENARIO)]
    display(SVG(distribution_chart(selected)))
    display(SVG(distribution_chart(selected, temporal=True)))
    for g in selected:
        print(g['database'], '20 misure ms:', g['values_ms'])

else:
    print('Operazione disabilitata: MOSTRA_APPROFONDIMENTI=False.')


In [ ]:
table('queries', ['database','dataset_size','query_id','index_state','lag1_descriptive','first_half_mean_ms','last_half_mean_ms','excluded_observations'])

## Selettività e crescita

Q1 è un lookup su PK, Q2/Q3 sono conteggi filtrati (non trasferimenti di tutti i record), Q4 aggrega l'intera relazione. La baseline ha soltanto PK; lo scenario secondary aggiunge gli indici approvati. I piani originali rimangono la fonte per l'accesso effettivamente scelto dal planner.

I rapporti di tempo vengono affiancati al rapporto N e, per Q2/Q3, al rapporto dei record corrispondenti. Il rapporto tempo/matches è solo una normalizzazione descrittiva, non un costo unitario causale. Prefissi, cache, concorrenza interna e selettività possono cambiare insieme. I grafici usano assi logaritmici; le linee collegano osservazioni, non regressioni.


In [ ]:
table('selection', ['dataset_size','Q2_rows','Q2_selectivity_pct','Q3_rows','Q3_selectivity_pct','Q4_groups'])
table('growth', ['database','query_id','index_state','from_size','to_size','row_ratio','mean_time_ratio','time_ratio_over_row_ratio','matching_row_ratio','time_ratio_over_matching_ratio'])

In [ ]:
if MOSTRA_APPROFONDIMENTI is True:
    for query, state in [('Q1','primary_only'),('Q2','primary_only'),('Q3','primary_only'),('Q4','primary_only'),('Q2','secondary'),('Q3','secondary')]:
        display(SVG(growth_chart(analysis['queries'], query, state)))

else:
    print('Operazione disabilitata: MOSTRA_APPROFONDIMENTI=False.')


## Indici: beneficio e costo

Speedup = media baseline / media indicizzata. Risparmio = differenza delle medie; riduzione percentuale rispetto alla baseline. Ammortamento **ipotetico** per una sola query e il relativo indice: ceil(costo costruzione / risparmio per query), soltanto se il risparmio è positivo. Non è una soglia di workload misurata: esclude manutenzione, nuove scritture, occupazione disco, cache e variabilità futura. Costruzione e ingestione sono singoli eventi, senza stima della loro incertezza.


In [ ]:
table('index_benefits', ['database','dataset_size','query_id','before_mean_ms','after_mean_ms','speedup','reduction_pct','build_ms_single_event','saved_ms_per_query','build_over_saving_queries','break_even_queries_hypothetical'])
table('single_events', ['database','dataset_size','event','query_id','n','mean_ms','rows_per_second','sample_stddev_ms','uncertainty'])

## PostgreSQL / MongoDB: query indicizzate FULL

Le sessioni sono distinte: nessun appaiamento per numero di iterazione. Si confrontano medie, mediane, dispersione e punti. La differenza delle medie divisa per la radice della media delle varianze è una misura descrittiva della sua grandezza rispetto alla dispersione interna; **non** un test, una probabilità o una stima della variabilità fra campagne. Anche sovrapposizione dei range/IQR non dimostra uguaglianza o assenza di differenza.

La frazione delle 400 ricombinazioni con tempo MongoDB inferiore usa sempre gli stessi 40 dati: **non sono 400 prove indipendenti**. Nel FULL MongoDB ha medie e mediane inferiori in entrambe le query; per Q2 il divario è piccolo rispetto alla dispersione, per Q3 è più evidente anche nelle mediane. L'osservazione non basta a generalizzare ad altre sessioni o macchine.


In [ ]:
if MOSTRA_APPROFONDIMENTI is True:
    table('pg_mongo_full', list(analysis['pg_mongo_full'][0]))
    for query in ('Q2','Q3'):
        groups = [g for g in analysis['queries'] if g['dataset_size']==5819079 and g['index_state']=='secondary' and g['query_id']==query and g['database'] in ('postgres','mongo')]
        display(SVG(distribution_chart(groups)))
        display(SVG(distribution_chart(groups, temporal=True)))
else:
    print('Operazione disabilitata: MOSTRA_APPROFONDIMENTI=False.')


## Interpretazione e limiti

**Evidenze misurate:** statistiche dei raw, conteggi e selettività, rapporti osservati, tempi singoli di ingestione/indici. Consultare le tabelle per motore, dimensione e classe di query: una graduatoria unica nasconderebbe carichi diversi.

**Interpretazioni plausibili:** differenze di accesso agli indici, cache e implementazione delle aggregazioni possono contribuire ai tempi; queste campagne non isolano causalmente tali fattori. Le politiche di commit e atomicità differiscono fra motori: stesso batch non equivale a identico lavoro o garanzie.

**Limiti:** ordine fisso dei motori/scenari, nessuna randomizzazione o replica indipendente delle sessioni, cache non svuotate, ambiente locale Docker/WSL2, versioni e carichi non isolati, una sola ingestione/costruzione per combinazione. Nessun intervallo di confidenza per n=1; SD assente non significa assenza di variabilità. Nessuna esclusione degli estremi o inferenza da queste sole 20 ripetizioni.


## Esportazione facoltativa dei soli derivati

Disabilitata per default. Una nuova directory UUID in `results/statistical_analysis/` conserva CSV, serie complete e mappa SHA-256 delle fonti. Prima di scrivere si verifica che le fonti non siano cambiate. Non vengono sovrascritti file delle campagne approvate.


In [ ]:
ESPORTA_DERIVATI = False
if ESPORTA_DERIVATI:
    print(export_analysis(analysis))